In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
!nvidia-smi

Mon Feb 23 20:17:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
REPO_URL = "https://github.com/SpaceSpiff20/daniel-splits.git"
REPO_DIR = "daniel-splits"

In [4]:
!git clone {REPO_URL}
%cd {REPO_DIR}

Cloning into 'daniel-splits'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 64 (delta 21), reused 57 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 319.89 KiB | 1.75 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/daniel-splits


In [5]:
# switch to daniel_splits branch
!git checkout daniel_splits
!git pull

Branch 'daniel_splits' set up to track remote branch 'daniel_splits' from 'origin'.
Switched to a new branch 'daniel_splits'
Already up to date.


In [6]:
!ls data/splits_out/superglue_boolq

manifest.json	  N16_seed0.jsonl   N256_seed1.jsonl  N32_seed2.jsonl
N128_seed0.jsonl  N16_seed1.jsonl   N256_seed2.jsonl  N64_seed0.jsonl
N128_seed1.jsonl  N16_seed2.jsonl   N32_seed0.jsonl   N64_seed1.jsonl
N128_seed2.jsonl  N256_seed0.jsonl  N32_seed1.jsonl   N64_seed2.jsonl


In [7]:
# Install libraries
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers datasets accelerate peft evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00


In [8]:
from pathlib import Path

SPLITS_DIR = Path("data/splits_out/superglue_boolq")

assert SPLITS_DIR.exists(), f"Missing splits dir: {SPLITS_DIR}"
print("Found split files:", sorted([p.name for p in SPLITS_DIR.glob("*.jsonl")])[:10])

Found split files: ['N128_seed0.jsonl', 'N128_seed1.jsonl', 'N128_seed2.jsonl', 'N16_seed0.jsonl', 'N16_seed1.jsonl', 'N16_seed2.jsonl', 'N256_seed0.jsonl', 'N256_seed1.jsonl', 'N256_seed2.jsonl', 'N32_seed0.jsonl']


In [9]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [14]:
!pip -q install bitsandbytes

In [13]:
# Load Qwen3-8B
!pip -q install -U bitsandbytes accelerate transformers

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cuda:0",            # keep everything on the single T4
    quantization_config=bnb_config, # <-- this is what actually reduces VRAM
    attn_implementation="eager",
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Optional: reduce KV-cache memory spikes for long prompts
model.config.use_cache = False

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 159.4 MB/s eta 0:00:00


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

ImportError: cannot import name 'merge_with_config_defaults' from 'transformers.utils.generic' (/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py)

In [ ]:
import re
from datasets import load_dataset
from pathlib import Path
import torch